<a href="https://colab.research.google.com/github/harshal3141592/tower_bloxx_RL/blob/main/tower_bloxx_rl_CNNv5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tower Bloxx / City Bloxx puzzle — solving it with reinforcement learning

This notebook is split into two parts that are meant to run in **separate kernel sessions**,
with a runtime restart in between:

- **Part A (below, cells 1-3)**: installs only `ortools`, solves the 3x3 and 4x4 puzzles exactly
  with CP-SAT, and saves the results to disk (`expert_solutions_blue.pkl`). Nothing here ever imports
  torch, stable-baselines3, gymnasium, or TensorFlow.
- **Part B (everything after the restart instruction)**: installs the RL stack, loads the cached
  CP-SAT results from disk, and does behavior-cloning pretraining + PPO training.

Why split it this way: repeated Colab crashes during the CP-SAT solve turned out to be memory
contention, not a bug in the CP-SAT model. Colab's kernel auto-imports TensorFlow on startup
(you never call it yourself), and once the RL stack (torch, stable-baselines3, gymnasium) is also
loaded in the same process, CP-SAT's search memory is competing with all of that baseline for a
tight RAM budget. CP-SAT never needs any of the RL libraries — running it in its own minimal
kernel, saving the result, then restarting before the heavy libraries ever load removes that
contention entirely, rather than just tuning worker counts and hoping.

**How to run this notebook:**
1. Run cells 1-3 below (Part A).
2. When you see "STOP — restart the runtime now", go to **Runtime → Restart runtime** (not
   "Restart and run all"). This clears the kernel's memory but keeps your files on disk, including
   `expert_solutions_blue.pkl`.
3. Continue running from the "Part B" markdown cell onward.


In [ ]:
!pip install -q ortools


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.


In [ ]:
import os
from ortools.sat.python import cp_model

VALUES = [0, 10, 20, 30, 40]           # empty, B, R, G, Y  (same scale as COLOR_VALUE)
REQ_IDS = {1: [], 2: [1], 3: [1, 2], 4: [1, 2, 3]}  # color id -> required color ids


def _bool_and(model, a, b, name):
    """Linear AND-reification for two BoolVars (standard 3-inequality trick)."""
    m = model.NewBoolVar(name)
    model.Add(m <= a)
    model.Add(m <= b)
    model.Add(m >= a + b - 1)
    return m


def solve_exact(grid_size, T, time_limit=60, initial_color=1):
    """
    Exact time-indexed CP-SAT solve of the Tower Bloxx puzzle.

    Models the SAME rule as the Gym env: placing color c at cell i at time t requires
    that, at time t-1 (i.e. *before* this placement), i's neighbors already include every
    color in REQUIREMENTS[c] -- checked only at the moment of placement, not maintained
    afterward. `T` is the number of placement actions allowed (a budget, like max_steps). The board starts
    all-Blue by default (initial_color=1); the optimum is the same as from an empty board.

    Returns a dict with the action sequence (in build order), the final grid, and the score,
    or None if no feasible/optimal solution was found within time_limit.
    """
    n = grid_size
    cells = [(r, c) for r in range(n) for c in range(n)]
    idx = {rc: i for i, rc in enumerate(cells)}
    n_cells = len(cells)

    def neighbors(r, c):
        out = []
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n:
                out.append(idx[(nr, nc)])
        return out

    NEIGH = [neighbors(r, c) for r, c in cells]

    model = cp_model.CpModel()
    # is_color[t][i][k] = 1 iff cell i has color k (0=empty,1=B,2=R,3=G,4=Y) at time t
    is_color = [[[model.NewBoolVar(f'c_{t}_{i}_{k}') for k in range(5)]
                 for i in range(n_cells)] for t in range(T + 1)]

    for t in range(T + 1):
        for i in range(n_cells):
            model.AddExactlyOne(is_color[t][i])

    for i in range(n_cells):                       # start state: initial_color=1 -> every cell already Blue (free); 0 -> empty board
        model.Add(is_color[0][i][initial_color] == 1)

    for t in range(1, T + 1):                        # can never revert to empty
        for i in range(n_cells):
            model.Add(is_color[t][i][0] <= is_color[t - 1][i][0])

    changed = [[model.NewBoolVar(f'chg_{t}_{i}') for i in range(n_cells)] for t in range(T)]

    for ti in range(T):
        t = ti + 1
        for i in range(n_cells):
            overlap = [_bool_and(model, is_color[t][i][k], is_color[t - 1][i][k], f'ov_{t}_{i}_{k}')
                       for k in range(5)]
            model.Add(sum(overlap) == 1).OnlyEnforceIf(changed[ti][i].Not())
            model.Add(sum(overlap) == 0).OnlyEnforceIf(changed[ti][i])

        model.Add(sum(changed[ti]) <= 1)              # at most one placement per timestep

        for i in range(n_cells):
            for c in range(1, 5):
                for r in REQ_IDS[c]:
                    neigh_vars = [is_color[t - 1][j][r] for j in NEIGH[i]]
                    if neigh_vars:
                        model.Add(sum(neigh_vars) >= 1).OnlyEnforceIf([is_color[t][i][c], changed[ti][i]])
                    else:
                        model.Add(is_color[t][i][c] == 0).OnlyEnforceIf(changed[ti][i])

    score_terms = [VALUES[k] * is_color[T][i][k] for i in range(n_cells) for k in range(5)]
    model.Maximize(sum(score_terms))

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    # Colab's free tier typically has ~2 vCPUs and limited RAM; each CP-SAT search worker
    # holds its own copy of the search state, so asking for more workers than you have cores
    # (or than your RAM budget allows) risks an out-of-memory kernel crash, especially on the
    # larger 4x4 solve. 2 workers is a conservative, evidenced-safe default on free-tier Colab
    # -- raise it only if you know you have more cores/RAM to spare.
    solver.parameters.num_search_workers = min(2, os.cpu_count() or 1)

    # Cheap, fast sanity check that costs nothing at solve time: catches a genuinely malformed
    # model (bad domains, malformed constraints) *before* handing it to the C++ solver, rather
    # than a crash or a confusing message from inside Solve(). This is NOT a fix for OOM kills
    # (which show up as a silent kernel restart, not a Python error) -- it only guards against
    # the model itself being invalid, a different failure mode.
    validation_error = model.Validate()
    if validation_error:
        print(f"Model validation error: {validation_error}")
        return None

    status = solver.Solve(model)
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return None

    actions = []
    for ti in range(T):
        t = ti + 1
        for i in range(n_cells):
            if solver.Value(changed[ti][i]):
                for c in range(1, 5):
                    if solver.Value(is_color[t][i][c]):
                        actions.append((cells[i], c))     # ((r, c), color_id)

    final_grid = [[None] * n for _ in range(n)]
    for i, (r, c) in enumerate(cells):
        for k in range(5):
            if solver.Value(is_color[T][i][k]):
                final_grid[r][c] = k
    score = sum(VALUES[final_grid[r][c]] for r in range(n) for c in range(n))
    return {"status": solver.StatusName(status), "actions": actions, "final_grid": final_grid, "score": score}


def render_solution(res):
    chars = {0: '.', 1: 'B', 2: 'R', 3: 'G', 4: 'Y'}
    for row in res["final_grid"]:
        print(' '.join(chars[k] for k in row))
    print("Score:", res["score"], f"({len(res['actions'])} placements, status={res['status']})")


# 3x3 solves in a few seconds; 4x4 is the largest size CP-SAT handles exactly in
# reasonable time (~1 min here) -- both become curriculum stages for the 5x5 agent below.
#
# If this cell crashes your Colab session (RAM exhaustion, not a Python error -- check the
# RAM gauge top-right before it resets), the 4x4 solve is the likely cause: it's the larger
# search. Lower time_limit further (e.g. 45) and/or set expert_solutions[4] = None below --
# the rest of the notebook degrades gracefully to a 3x3-only curriculum (see the behavior-
# cloning cell), it just warm-starts the 5x5 backbone a bit less.
expert_solutions = {}
expert_solutions[3] = solve_exact(grid_size=3, T=12, time_limit=45)
print("--- 3x3 optimal ---")
render_solution(expert_solutions[3])

expert_solutions[4] = solve_exact(grid_size=4, T=18, time_limit=90)
print("\n--- 4x4 optimal ---")
if expert_solutions[4] is not None:
    render_solution(expert_solutions[4])
else:
    print("(skipped/unavailable -- downstream cells fall back to a 3x3-only curriculum)")


import pickle

with open("expert_solutions_blue.pkl", "wb") as f:
    pickle.dump(expert_solutions, f)
print("\nSaved expert_solutions_blue.pkl to disk -- safe to restart the runtime now.")


--- 3x3 optimal ---
B Y G
Y R Y
G Y R
Score: 270 (12 placements, status=OPTIMAL)

--- 4x4 optimal ---
R G Y G
G Y Y G
Y R G Y
B G Y R
Score: 490 (18 placements, status=OPTIMAL)

Saved expert_solutions_blue.pkl to disk -- safe to restart the runtime now.


---
## ⏸ STOP — restart the runtime now

Go to **Runtime → Restart runtime** in the Colab menu. This clears TensorFlow and everything else
Colab loaded automatically, without deleting `expert_solutions_blue.pkl` from disk.

Then continue with the cells below (Part B).

---


## Part B: environment, CNN policy, and training

Everything from here on installs and uses the RL stack (torch, stable-baselines3, gymnasium). It
loads the CP-SAT results computed in Part A from `expert_solutions_blue.pkl` instead of re-solving them
in this process.


In [ ]:
!pip install -q stable-baselines3 sb3-contrib gymnasium shimmy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 8.5 MB/s eta 0:00:00


## 1. Environment

A custom Gymnasium environment. Actions are encoded as `cell_index * 4 + color_index`, plus one extra
**STOP** action (index `n_cells * 4`) that ends the episode.

`action_masks()` returns which actions are currently legal, and it is fed to `MaskablePPO` so the agent
only ever samples valid moves. Two details matter a lot for learning:

- **No-op placements are masked out.** Re-placing a cell's current color is technically legal under the
  rules and earns reward 0. Left in, it gives a deterministic policy a fixed point to get stuck on: the
  board stops changing, so the observation stops changing, so the same action is chosen forever. (This
  is what produced a half-empty 5x5 board and a score of 270 in an earlier run.) Every non-STOP action
  now changes the board.
- **STOP is the only way to "do nothing".** Since every placement must change something, the agent needs a
  way to say "the board is finished" instead of being forced to tear it down in the last steps.
- **The board starts all Blue.** Blue has no requirement, so an all-Blue board is always reachable and is
  the natural floor (5x5: 250 points). Starting there costs nothing in terms of the best reachable score
  (CP-SAT gives the same optimum from an all-Blue start on 3x3 and 4x4: 270 and 490) and it removes the
  first ~25 placements from what the agent has to plan, so it can focus on the upgrade schedule.
  Pass `start_blue=False` to get the original empty board.


In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces

COLORS = ['empty', 'B', 'R', 'G', 'Y']
COLOR_VALUE = {'empty': 0, 'B': 10, 'R': 20, 'G': 30, 'Y': 40}
COLOR_IDX = {c: i for i, c in enumerate(COLORS)}
N_COLORS = len(COLORS)
BUILD_COLORS = ['B', 'R', 'G', 'Y']
N_BUILD_COLORS = len(BUILD_COLORS)

REQUIREMENTS = {
    'B': set(),
    'R': {'B'},
    'G': {'B', 'R'},
    'Y': {'B', 'R', 'G'},
}


class TowerBloxxEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, grid_size=5, max_steps=60, start_blue=True):
        super().__init__()
        self.n = grid_size
        self.max_steps = max_steps
        self.start_blue = start_blue      # every cell begins as Blue (score floor n*n*10)
        self.n_cells = grid_size * grid_size
        self.STOP = self.n_cells * N_BUILD_COLORS          # extra action: end the episode
        self.action_space = spaces.Discrete(self.n_cells * N_BUILD_COLORS + 1)
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(self.n_cells * N_COLORS + 1,), dtype=np.float32
        )
        self.grid = np.zeros((self.n, self.n), dtype=np.int64)
        self.step_count = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.grid = np.full((self.n, self.n), COLOR_IDX['B'] if self.start_blue else 0, dtype=np.int64)
        self.step_count = 0
        return self._get_obs(), {}

    def _neighbors(self, r, c):
        coords = []
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < self.n and 0 <= nc < self.n:
                coords.append((nr, nc))
        return coords

    def _neighbor_colors(self, r, c):
        return {COLORS[self.grid[nr, nc]] for nr, nc in self._neighbors(r, c)}

    def _is_legal(self, r, c, color):
        return REQUIREMENTS[color].issubset(self._neighbor_colors(r, c))

    def action_masks(self):
        mask = np.zeros(self.n_cells * N_BUILD_COLORS + 1, dtype=bool)
        for idx in range(self.n_cells):
            r, c = divmod(idx, self.n)
            for ci, color in enumerate(BUILD_COLORS):
                # legal AND actually changes the cell (no-op re-placements are masked out)
                if self._is_legal(r, c, color) and COLOR_IDX[color] != self.grid[r, c]:
                    mask[idx * N_BUILD_COLORS + ci] = True
        mask[self.STOP] = True
        return mask

    def step(self, action):
        if int(action) == self.STOP:
            return self._get_obs(), 0.0, True, False, {}

        cell_idx, color_idx = divmod(int(action), N_BUILD_COLORS)
        r, c = divmod(cell_idx, self.n)
        color = BUILD_COLORS[color_idx]

        old_value = COLOR_VALUE[COLORS[self.grid[r, c]]]
        if self._is_legal(r, c, color):
            self.grid[r, c] = COLOR_IDX[color]
            new_value = COLOR_VALUE[color]
            reward = float(new_value - old_value)
        else:
            # Should rarely fire when action masking is used correctly.
            reward = -5.0

        self.step_count += 1
        terminated = False
        truncated = self.step_count >= self.max_steps
        return self._get_obs(), reward, terminated, truncated, {}

    def _get_obs(self):
        onehot = np.zeros((self.n_cells, N_COLORS), dtype=np.float32)
        onehot[np.arange(self.n_cells), self.grid.flatten()] = 1.0
        return np.concatenate([onehot.flatten(), [self.step_count / self.max_steps]]).astype(np.float32)

    @property
    def score(self):
        return int(sum(COLOR_VALUE[COLORS[v]] for v in self.grid.flatten()))

    def render(self):
        for row in self.grid:
            print(' '.join(COLORS[v][0] if v else '.' for v in row))
        print('Score:', self.score)

## 2. Sanity check with random legal moves

In [ ]:
env = TowerBloxxEnv(grid_size=5, max_steps=60)
obs, _ = env.reset()
rng = np.random.default_rng(0)
done = False
while not done:
    mask = env.action_masks()
    legal_actions = np.flatnonzero(mask)
    legal_actions = legal_actions[legal_actions != env.STOP]   # random baseline: never stop early
    action = rng.choice(legal_actions)
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

env.render()

R B G R R
B R Y R R
R B R G B
R B R B R
B R B B B
Score: 440


## 3. A convolutional feature extractor

The plain `MlpPolicy` treats the board as a flat 126-length vector with no notion of "which cells are adjacent." Since every rule in this game is a 4-neighbor adjacency rule, a small CNN is a much better inductive bias — it should learn faster and generalize better across grid sizes. This custom extractor reshapes the flat one-hot observation back into a `(N_COLORS, n, n)` image, runs it through a couple of conv layers, and concatenates the step-count scalar back in.

In [ ]:
import torch
import torch.nn as nn
from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

class GridCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space, grid_size=5, n_colors=N_COLORS, features_dim=128):
        super().__init__(observation_space, features_dim)
        self.grid_size = grid_size
        self.n_colors = n_colors
        self.cnn = nn.Sequential(
            nn.Conv2d(n_colors, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Flatten(),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, n_colors, grid_size, grid_size)
            n_flatten = self.cnn(dummy).shape[1]
        self.linear = nn.Sequential(nn.Linear(n_flatten + 1, features_dim), nn.ReLU())

    def forward(self, observations):
        grid_flat, step_frac = observations[:, :-1], observations[:, -1:]
        batch = grid_flat.shape[0]
        # stored as (n_cells, n_colors) row-major -> reshape then move colors to channel dim
        grid = grid_flat.view(batch, self.grid_size, self.grid_size, self.n_colors).permute(0, 3, 1, 2)
        feats = self.cnn(grid)
        return self.linear(torch.cat([feats, step_frac], dim=1))

policy_kwargs = dict(
    features_extractor_class=GridCNN,
    features_extractor_kwargs=dict(grid_size=5, n_colors=N_COLORS, features_dim=128),
)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 4. Exact expert demonstrations (loaded from Part A)

Loads the CP-SAT solutions computed in Part A (`expert_solutions_blue.pkl`). If you skipped the
restart and are running everything in a single session, this falls back to solving inline here --
you lose the memory-isolation benefit, but it still works.


In [ ]:
import os
import pickle

if os.path.exists("expert_solutions_blue.pkl"):
    with open("expert_solutions_blue.pkl", "rb") as f:
        expert_solutions = pickle.load(f)
    print("Loaded expert_solutions_blue.pkl from Part A.")
else:
    print("expert_solutions_blue.pkl not found -- solving inline in this session as a fallback")
    print("(if this crashes, run Part A in its own fresh runtime instead -- see top of notebook).")
    import os
    from ortools.sat.python import cp_model

    VALUES = [0, 10, 20, 30, 40]           # empty, B, R, G, Y  (same scale as COLOR_VALUE)
    REQ_IDS = {1: [], 2: [1], 3: [1, 2], 4: [1, 2, 3]}  # color id -> required color ids


    def _bool_and(model, a, b, name):
        """Linear AND-reification for two BoolVars (standard 3-inequality trick)."""
        m = model.NewBoolVar(name)
        model.Add(m <= a)
        model.Add(m <= b)
        model.Add(m >= a + b - 1)
        return m


    def solve_exact(grid_size, T, time_limit=60, initial_color=1):
        """
        Exact time-indexed CP-SAT solve of the Tower Bloxx puzzle.

        Models the SAME rule as the Gym env: placing color c at cell i at time t requires
        that, at time t-1 (i.e. *before* this placement), i's neighbors already include every
        color in REQUIREMENTS[c] -- checked only at the moment of placement, not maintained
        afterward. `T` is the number of placement actions allowed (a budget, like max_steps). The board starts
    all-Blue by default (initial_color=1); the optimum is the same as from an empty board.

        Returns a dict with the action sequence (in build order), the final grid, and the score,
        or None if no feasible/optimal solution was found within time_limit.
        """
        n = grid_size
        cells = [(r, c) for r in range(n) for c in range(n)]
        idx = {rc: i for i, rc in enumerate(cells)}
        n_cells = len(cells)

        def neighbors(r, c):
            out = []
            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nr, nc = r + dr, c + dc
                if 0 <= nr < n and 0 <= nc < n:
                    out.append(idx[(nr, nc)])
            return out

        NEIGH = [neighbors(r, c) for r, c in cells]

        model = cp_model.CpModel()
        # is_color[t][i][k] = 1 iff cell i has color k (0=empty,1=B,2=R,3=G,4=Y) at time t
        is_color = [[[model.NewBoolVar(f'c_{t}_{i}_{k}') for k in range(5)]
                     for i in range(n_cells)] for t in range(T + 1)]

        for t in range(T + 1):
            for i in range(n_cells):
                model.AddExactlyOne(is_color[t][i])

        for i in range(n_cells):                       # start state: initial_color=1 -> every cell already Blue (free); 0 -> empty board
            model.Add(is_color[0][i][initial_color] == 1)

        for t in range(1, T + 1):                        # can never revert to empty
            for i in range(n_cells):
                model.Add(is_color[t][i][0] <= is_color[t - 1][i][0])

        changed = [[model.NewBoolVar(f'chg_{t}_{i}') for i in range(n_cells)] for t in range(T)]

        for ti in range(T):
            t = ti + 1
            for i in range(n_cells):
                overlap = [_bool_and(model, is_color[t][i][k], is_color[t - 1][i][k], f'ov_{t}_{i}_{k}')
                           for k in range(5)]
                model.Add(sum(overlap) == 1).OnlyEnforceIf(changed[ti][i].Not())
                model.Add(sum(overlap) == 0).OnlyEnforceIf(changed[ti][i])

            model.Add(sum(changed[ti]) <= 1)              # at most one placement per timestep

            for i in range(n_cells):
                for c in range(1, 5):
                    for r in REQ_IDS[c]:
                        neigh_vars = [is_color[t - 1][j][r] for j in NEIGH[i]]
                        if neigh_vars:
                            model.Add(sum(neigh_vars) >= 1).OnlyEnforceIf([is_color[t][i][c], changed[ti][i]])
                        else:
                            model.Add(is_color[t][i][c] == 0).OnlyEnforceIf(changed[ti][i])

        score_terms = [VALUES[k] * is_color[T][i][k] for i in range(n_cells) for k in range(5)]
        model.Maximize(sum(score_terms))

        solver = cp_model.CpSolver()
        solver.parameters.max_time_in_seconds = time_limit
        # Colab's free tier typically has ~2 vCPUs and limited RAM; each CP-SAT search worker
        # holds its own copy of the search state, so asking for more workers than you have cores
        # (or than your RAM budget allows) risks an out-of-memory kernel crash, especially on the
        # larger 4x4 solve. 2 workers is a conservative, evidenced-safe default on free-tier Colab
        # -- raise it only if you know you have more cores/RAM to spare.
        solver.parameters.num_search_workers = min(2, os.cpu_count() or 1)

        # Cheap, fast sanity check that costs nothing at solve time: catches a genuinely malformed
        # model (bad domains, malformed constraints) *before* handing it to the C++ solver, rather
        # than a crash or a confusing message from inside Solve(). This is NOT a fix for OOM kills
        # (which show up as a silent kernel restart, not a Python error) -- it only guards against
        # the model itself being invalid, a different failure mode.
        validation_error = model.Validate()
        if validation_error:
            print(f"Model validation error: {validation_error}")
            return None

        status = solver.Solve(model)
        if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
            return None

        actions = []
        for ti in range(T):
            t = ti + 1
            for i in range(n_cells):
                if solver.Value(changed[ti][i]):
                    for c in range(1, 5):
                        if solver.Value(is_color[t][i][c]):
                            actions.append((cells[i], c))     # ((r, c), color_id)

        final_grid = [[None] * n for _ in range(n)]
        for i, (r, c) in enumerate(cells):
            for k in range(5):
                if solver.Value(is_color[T][i][k]):
                    final_grid[r][c] = k
        score = sum(VALUES[final_grid[r][c]] for r in range(n) for c in range(n))
        return {"status": solver.StatusName(status), "actions": actions, "final_grid": final_grid, "score": score}


    def render_solution(res):
        chars = {0: '.', 1: 'B', 2: 'R', 3: 'G', 4: 'Y'}
        for row in res["final_grid"]:
            print(' '.join(chars[k] for k in row))
        print("Score:", res["score"], f"({len(res['actions'])} placements, status={res['status']})")


    # 3x3 solves in a few seconds; 4x4 is the largest size CP-SAT handles exactly in
    # reasonable time (~1 min here) -- both become curriculum stages for the 5x5 agent below.
    #
    # If this cell crashes your Colab session (RAM exhaustion, not a Python error -- check the
    # RAM gauge top-right before it resets), the 4x4 solve is the likely cause: it's the larger
    # search. Lower time_limit further (e.g. 45) and/or set expert_solutions[4] = None below --
    # the rest of the notebook degrades gracefully to a 3x3-only curriculum (see the behavior-
    # cloning cell), it just warm-starts the 5x5 backbone a bit less.
    expert_solutions = {}
    expert_solutions[3] = solve_exact(grid_size=3, T=12, time_limit=45)
    print("--- 3x3 optimal ---")
    render_solution(expert_solutions[3])

    expert_solutions[4] = solve_exact(grid_size=4, T=18, time_limit=90)
    print("\n--- 4x4 optimal ---")
    if expert_solutions[4] is not None:
        render_solution(expert_solutions[4])
    else:
        print("(skipped/unavailable -- downstream cells fall back to a 3x3-only curriculum)")


Loaded expert_solutions_blue.pkl from Part A.


## 5. Behavior-cloning dataset (with dihedral symmetry augmentation)

Each exact solve gives one optimal build order, but the puzzle's adjacency structure is symmetric
under the 8 transformations of the square (4 rotations x horizontal flip). Replaying the same solution
under each of those transforms gives 8 equally-valid, equally-optimal demonstrations for free, without
calling CP-SAT again — cheap data augmentation that also nudges the CNN toward rotation/reflection-
consistent features, which is exactly the kind of local, orientation-agnostic pattern (`does this cell
have a Blue+Red neighbor?`) the conv backbone should be learning anyway.


In [ ]:
def _transform_coords(r, c, n, k, flip):
    for _ in range(k):
        r, c = c, n - 1 - r          # rotate 90 deg
    if flip:
        c = n - 1 - c                # mirror
    return r, c


def build_bc_dataset(actions, n):
    """Replay one optimal action sequence under all 8 grid symmetries, recording
    (obs, action_mask, action) at every step -- the standard behavior-cloning format."""
    dataset = []
    for k in range(4):
        for flip in (False, True):
            env = TowerBloxxEnv(grid_size=n, max_steps=len(actions))
            env.reset()
            for (r, c), color_id in actions:
                tr, tc = _transform_coords(r, c, n, k, flip)
                color = COLORS[color_id]
                cell_idx = tr * n + tc
                action = cell_idx * N_BUILD_COLORS + BUILD_COLORS.index(color)
                obs_before = env._get_obs()
                mask_before = env.action_masks()
                env.step(action)
                dataset.append((obs_before, mask_before, action))
            # after the last placement the expert is done: teach the STOP action too
            dataset.append((env._get_obs(), env.action_masks(), env.STOP))
    return dataset


bc_datasets = {n: build_bc_dataset(expert_solutions[n]["actions"], n)
               for n in (3, 4) if expert_solutions.get(n) is not None}
for n, ds in bc_datasets.items():
    print(f"{n}x{n}: {len(ds)} demonstration steps (base steps + 1 STOP, x 8 symmetries)")


3x3: 104 demonstration steps (base steps + 1 STOP, x 8 symmetries)
4x4: 152 demonstration steps (base steps + 1 STOP, x 8 symmetries)


## 6. Pretrain the CNN backbone via behavior cloning, then curriculum-transfer to 5x5

We pretrain on 3x3, transfer just the convolutional backbone's weights into a fresh 4x4 model and
keep refining on 4x4 data, then transfer that backbone into the 5x5 PPO model below. The conv layers
(`GridCNN.cnn`) are the only weights that transfer cleanly across grid sizes — they're just 3x3 kernels
applied everywhere, independent of board size. The bottleneck `Linear` layer and the actor/value heads
depend on the specific grid size (flattened feature count, number of actions), so those stay freshly
initialized at each stage; only the local-pattern detectors carry over. This is standard curriculum-
learning by problem size: an easy version of the task (short horizon, exact solutions available) warm-
starts a harder one (5x5, PPO) instead of PPO starting from a random backbone.


In [ ]:
import torch
from sb3_contrib import MaskablePPO
from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy


def behavior_clone(grid_size, dataset, init_cnn_state=None, epochs=200, lr=1e-3):
    env = TowerBloxxEnv(grid_size=grid_size, max_steps=1)
    pk = dict(features_extractor_class=GridCNN,
              features_extractor_kwargs=dict(grid_size=grid_size, n_colors=N_COLORS, features_dim=128))
    model = MaskablePPO(MaskableActorCriticPolicy, env, policy_kwargs=pk, verbose=0, seed=0)
    policy = model.policy
    if init_cnn_state is not None:
        policy.features_extractor.cnn.load_state_dict(init_cnn_state)

    obs = torch.tensor(np.array([d[0] for d in dataset]), dtype=torch.float32)
    mask = np.array([d[1] for d in dataset])
    actions = torch.tensor(np.array([d[2] for d in dataset]), dtype=torch.long)

    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    for epoch in range(epochs):
        values, log_prob, entropy = policy.evaluate_actions(obs, actions, action_masks=mask)
        loss = -log_prob.mean() - 0.01 * entropy.mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
        if epoch % 50 == 0 or epoch == epochs - 1:
            with torch.no_grad():
                pred = policy.get_distribution(obs, action_masks=mask).get_actions(deterministic=True)
                acc = (pred == actions).float().mean().item()
            print(f"  [{grid_size}x{grid_size}] epoch {epoch:3d}  loss {loss.item():.4f}  train_acc {acc:.3f}")
    return policy.features_extractor.cnn.state_dict()


print("Stage 1: behavior cloning on 3x3")
cnn_state = behavior_clone(3, bc_datasets[3], init_cnn_state=None)

if 4 in bc_datasets:
    print("Stage 2: behavior cloning on 4x4 (backbone warm-started from 3x3)")
    cnn_state = behavior_clone(4, bc_datasets[4], init_cnn_state=cnn_state)
else:
    print("Skipping 4x4 stage (no 4x4 solution available) -- using 3x3-only backbone.")

print("\nPretrained conv backbone ready to transfer into the 5x5 PPO model.")


Stage 1: behavior cloning on 3x3
  [3x3] epoch   0  loss 2.7760  train_acc 0.394


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  [3x3] epoch  50  loss 1.4569  train_acc 0.625
  [3x3] epoch 100  loss 0.9012  train_acc 0.702
  [3x3] epoch 150  loss 0.6489  train_acc 0.731
  [3x3] epoch 199  loss 0.4651  train_acc 0.788
Stage 2: behavior cloning on 4x4 (backbone warm-started from 3x3)
  [4x4] epoch   0  loss 3.1920  train_acc 0.257
  [4x4] epoch  50  loss 1.9113  train_acc 0.388
  [4x4] epoch 100  loss 1.5737  train_acc 0.414
  [4x4] epoch 150  loss 1.2364  train_acc 0.592
  [4x4] epoch 199  loss 0.8783  train_acc 0.770

Pretrained conv backbone ready to transfer into the 5x5 PPO model.


## 7. Train MaskablePPO on 5x5 (warm-started backbone, parallel envs)

Three changes from before, on top of the CNN policy:
1. **Warm start**: the 5x5 model's conv backbone (`GridCNN.cnn`) is initialized from the 3x3-to-4x4
   behavior-cloning curriculum above, instead of random weights. The linear bottleneck and actor/value
   heads are still fresh (they're sized for this grid's action space), but PPO no longer has to learn
   "does this cell have a Blue+Red neighbor" from scratch via reward alone.
2. **Parallel environments** (`SubprocVecEnv`, one process per CPU core): rollout collection is the
   dominant cost of `model.learn()`, and it's embarrassingly parallel across independent envs, so this
   should cut wall-clock time roughly in proportion to core count. Colab's free CPU runtime typically
   gives 2 cores, so expect ~1.5-2x, not a dramatic speedup on its own — it's the warm start that should
   do most of the work in getting closer to 810 faster. If `SubprocVecEnv` misbehaves in your Colab
   session (it occasionally does under certain notebook/multiprocessing configurations), fall back to
   `DummyVecEnv` by flipping the flag below; correctness doesn't depend on which one you use.
3. Because of the warm start, you can likely get a useful read on progress with **fewer total
   timesteps** than before — `total_timesteps` below is left at 2,000,000 as a safe default, but check
   the eval curve early and stop (or extend) based on how it's trending rather than assuming you need
   the full budget.


In [ ]:
import os
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.common.maskable.callbacks import MaskableEvalCallback
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv

def mask_fn(env):
    return env.action_masks()

GRID_SIZE = 5
MAX_STEPS = 60       # all-Blue start: 3x3 needs 12 placements, 4x4 needs 18, so 5x5 should need ~30-40
USE_WARM_START = True   # set False to A/B against training from a random backbone
USE_SUBPROC = True   # flip to False if SubprocVecEnv gives you trouble in this Colab session

def make_env():
    def _init():
        env = TowerBloxxEnv(grid_size=GRID_SIZE, max_steps=MAX_STEPS)
        return ActionMasker(env, mask_fn)
    return _init

n_envs = max(1, os.cpu_count() or 1)
print(f"Using {n_envs} parallel environment(s) ({'SubprocVecEnv' if USE_SUBPROC else 'DummyVecEnv'})")

VecEnvCls = SubprocVecEnv if USE_SUBPROC and n_envs > 1 else DummyVecEnv
train_env = VecEnvCls([make_env() for _ in range(n_envs)])
eval_env = make_env()()   # single env is fine for periodic evaluation

eval_callback = MaskableEvalCallback(
    eval_env,
    best_model_save_path="./best_model",
    log_path="./eval_logs",
    eval_freq=max(20_000 // n_envs, 1),   # keep the same *total-step* eval cadence under vectorization
    n_eval_episodes=5,
    deterministic=True,
)

model = MaskablePPO(
    MaskableActorCriticPolicy,
    train_env,
    policy_kwargs=policy_kwargs,
    gamma=0.995,
    ent_coef=0.01,
    n_steps=2048 // n_envs if n_envs > 1 else 2048,   # keep rollout buffer size roughly comparable
    batch_size=256,
    verbose=1,
    seed=0,
)

# Warm-start the conv backbone from the 3x3->4x4 behavior-cloning curriculum.
if USE_WARM_START:
    model.policy.features_extractor.cnn.load_state_dict(cnn_state)
    print("Warm-started 5x5 conv backbone from behavior-cloned weights.")
else:
    print("Training from a random backbone (no warm start).")

model.learn(total_timesteps=2_000_000, callback=eval_callback)
model.save("tower_bloxx_ppo_cnn")


Using 2 parallel environment(s) (SubprocVecEnv)
Using cpu device
Warm-started 5x5 conv backbone from behavior-cloned weights.


/usr/local/lib/python3.13/dist-packages/stable_baselines3/common/callbacks.py:419: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.subproc_vec_env.SubprocVecEnv object at 0x7ff46f1127b0> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x7ff3be4a2990>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


-----------------------------
| time/              |      |
|    fps             | 739  |
|    iterations      | 1    |
|    time_elapsed    | 2    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 560         |
|    iterations           | 2           |
|    time_elapsed         | 7           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.011113817 |
|    clip_fraction        | 0.0484      |
|    clip_range           | 0.2         |
|    entropy_loss         | -3.69       |
|    explained_variance   | -0.00198    |
|    learning_rate        | 0.0003      |
|    loss                 | 1.6e+03     |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0204     |
|    value_loss           | 3.1e+03     |
-----------------------------------------
----------------------------------

/usr/local/lib/python3.13/dist-packages/sb3_contrib/common/maskable/evaluation.py:73: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Streaming output truncated to the last 5000 lines.
|    iterations      | 713     |
|    time_elapsed    | 3728    |
|    total_timesteps | 1460224 |
--------------------------------
------------------------------------------
| time/                   |              |
|    fps                  | 391          |
|    iterations           | 714          |
|    time_elapsed         | 3734         |
|    total_timesteps      | 1462272      |
| train/                  |              |
|    approx_kl            | 0.0026756006 |
|    clip_fraction        | 0.0187       |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.234       |
|    explained_variance   | 1            |
|    learning_rate        | 0.0003       |
|    loss                 | 1.87         |
|    n_updates            | 7130         |
|    policy_gradient_loss | -0.00411     |
|    value_loss           | 2.81         |
------------------------------------------
------------------------------------------


## 8. Evaluate the trained agent

Loads the **best** checkpoint the eval callback saved during training (not necessarily the final one — PPO can wobble), then does one deterministic rollout.

In [ ]:
best_model = MaskablePPO.load("./best_model/best_model")

watch_env = TowerBloxxEnv(grid_size=GRID_SIZE, max_steps=MAX_STEPS)
obs, _ = watch_env.reset()
done = False
while not done:
    mask = watch_env.action_masks()
    action, _ = best_model.predict(obs, action_masks=mask, deterministic=True)
    obs, reward, terminated, truncated, info = watch_env.step(action)
    done = terminated or truncated

watch_env.render()
print(f"\nAgent final score: {watch_env.score} (known optimal for the unconstrained-move version: 810)")

R G R Y G
G Y G Y R
Y R R Y G
G Y G B Y
R Y G Y R
Score: 750

Agent final score: 750 (known optimal for the unconstrained-move version: 810)


## 9. Greedy baseline (for comparison)

A simple heuristic: at each step, take whichever *legal* action increases score the most right now. This is a weak baseline (myopic, no lookahead) — your RL agent should beat it once trained, since RL can learn to sacrifice short-term value (e.g. placing throwaway Blues as scaffolding) for a better long-term outcome.

In [ ]:
def greedy_rollout(grid_size, max_steps):
    env = TowerBloxxEnv(grid_size=grid_size, max_steps=max_steps)
    obs, _ = env.reset()
    done = False
    while not done:
        mask = env.action_masks()
        best_action, best_gain = None, -1
        for a in np.flatnonzero(mask):
            if a == env.STOP:
                continue
            cell_idx, color_idx = divmod(int(a), N_BUILD_COLORS)
            r, c = divmod(cell_idx, env.n)
            gain = COLOR_VALUE[BUILD_COLORS[color_idx]] - COLOR_VALUE[COLORS[env.grid[r, c]]]
            if gain > best_gain:
                best_gain, best_action = gain, a
        if best_action is None or best_gain <= 0:
            best_action = env.STOP          # nothing improves the board: stop
        obs, reward, terminated, truncated, info = env.step(best_action)
        done = terminated or truncated
    return env

g_env = greedy_rollout(GRID_SIZE, MAX_STEPS)
g_env.render()

R G R G R
G R G R G
R G R G R
G R G R G
B G B G B
Score: 590


## 10. Validating against a known optimum

`kevindalmeijer/TowerBlocksOptimizer` already computed the *provably optimal* 5x5 score for the "Simple" scoring (B=1,R=2,G=3,Y=4): **81**. Scaled by 10x to match this notebook's point values (B=10,...,Y=40), that's an optimal score of **810** for the unconstrained-move-count version of the puzzle. Run your agent with a generous `max_steps` (e.g. 150–200) and see how close it gets to 810 — that gap is your headline result. If you want an exact CP-SAT ground truth for a *specific* move budget instead of the unconstrained case, clone that repo and adapt `cp_optimizer.py`, or install OR-Tools directly:

```
!pip install -q ortools
```

and build a time-indexed CP-SAT model (a boolean variable per cell/color/timestep) to solve the bounded-budget version exactly for small grids, then compare against the RL agent.

## 11. Where this connects to the peptide-design idea

This exact recipe — a graph of discrete positions, a physically-motivated local legality/energy rule between neighbors, and a learned policy trained to search that space — is structurally the same problem real protein/peptide sequence-design tools solve. `ProteinMPNN`, for instance, is a graph neural network that predicts amino acids position-by-position over a protein backbone graph, trained where this notebook's PPO agent is trained by trial-and-error reward. Two natural next steps if you want to build toward that:

1. **Swap the MLP policy for a graph neural network** that operates on the grid's adjacency structure directly, rather than a flattened one-hot vector. This is a closer architectural match to how ProteinMPNN represents structure, and it should generalize better across grid sizes.
2. **Try imitation learning as a second approach**: use OR-Tools to solve many small/random grids optimally, record the optimal move sequences, and train a network via supervised learning (cross-entropy on "which move did the optimal solver make here") to imitate the solver instead of learning from scratch via RL. This mirrors how ProteinMPNN itself is trained — supervised imitation of a known-good target (the native sequence), not RL.